<div style="
    border: 2px solid #1769aa;
    border-radius: 18px;
    padding: 34px 28px;
    background: linear-gradient(135deg, #eef7ff 0%, #ffffff 55%, #eefaf7 100%);
    text-align: center;
    box-shadow: 0 6px 20px rgba(20, 82, 130, 0.12);
">
    <div style="
        width: 78px;
        height: 78px;
        margin: 0 auto 18px auto;
        border-radius: 50%;
        display: flex;
        align-items: center;
        justify-content: center;
        background: linear-gradient(145deg, #0b5f9e, #2a91df);
        color: #ffffff;
        font-size: 24px;
        font-weight: 800;
        letter-spacing: 1px;
        box-shadow: 0 5px 14px rgba(15, 87, 138, 0.28);
    ">
        HNTT
    </div>

    <div style="font-size: 15px; color: #496579; letter-spacing: 1.6px; font-weight: 700;">
        ĐỒ ÁN CUỐI KỲ
    </div>

    <h1 style="
        margin: 10px 0 8px 0;
        color: #123b5d;
        font-size: 34px;
        line-height: 1.25;
    ">
        THUẬT TOÁN A* CHO ROBOT HÚT BỤI
    </h1>

    <p style="
        margin: 0 auto 24px auto;
        max-width: 760px;
        color: #526f84;
        font-size: 16px;
    ">
        Mô hình hóa bài toán làm sạch ma trận, tìm lộ trình tối ưu và tính tổng chi phí
        bằng thuật toán A*
    </p>

    <table style="
        margin: 0 auto;
        border-collapse: separate;
        border-spacing: 0 8px;
        font-size: 16px;
        color: #20394c;
        text-align: left;
    ">
        <tr>
            <td style="padding-right: 24px; font-weight: 700;">Người thực hiện</td>
            <td><b>Hoàng Ngọc Thủy Thương</b></td>
        </tr>
        <tr>
            <td style="padding-right: 24px; font-weight: 700;">Giảng viên</td>
            <td>TS. Nguyễn An Tế</td>
        </tr>
        <tr>
            <td style="padding-right: 24px; font-weight: 700;">Ngôn ngữ</td>
            <td>Python – Jupyter Notebook (.ipynb)</td>
        </tr>
        <tr>
            <td style="padding-right: 24px; font-weight: 700;">Thời gian</td>
            <td>Tháng 04/2026</td>
        </tr>
    </table>
</div>

---

### Giới thiệu bài toán

Một vùng hình chữ nhật được chia thành ma trận \(A_{m,n}\), với gốc tọa độ
\((1,1)\) đặt ở góc dưới bên trái. Mỗi ô có trạng thái `Dirty (D)` hoặc
`Clean (C)`. Robot có thể thực hiện các hành động:

- `MOVE_LEFT`, `MOVE_RIGHT`, `MOVE_UP`, `MOVE_DOWN`;
- `SUCK` để làm sạch ô đang đứng.

Sau mỗi hành động, chi phí bước được tính bởi:

\[
c = 1 + \left|D_{\text{còn lại sau hành động}}\right|.
\]

Mục tiêu là sử dụng thuật toán A* để tìm lộ trình làm sạch toàn bộ các ô dirty
với tổng chi phí nhỏ nhất.


In [ ]:

# Nếu máy chưa có thư viện, bỏ dấu # ở dòng dưới và chạy một lần.
# %pip install pandas openpyxl matplotlib ipywidgets

import heapq
import math
import random
from dataclasses import dataclass
from itertools import count
from pathlib import Path
from typing import FrozenSet, List, Optional, Tuple

import matplotlib.pyplot as plt


# ============================================================
# THÔNG TIN ĐỒ ÁN
# ============================================================
STUDENT_NAME = "Hoàng Ngọc Thủy Thương"
PROJECT_TITLE = "Đồ án cuối kỳ – Thuật toán A* cho Robot hút bụi"
LECTURER_NAME = "TS. Nguyễn An Tế"
SUBMISSION_TIME = "Tháng 04/2026"

import pandas as pd
from IPython.display import display


In [ ]:

Position = Tuple[int, int]
State = Tuple[Position, FrozenSet[Position]]

@dataclass(frozen=True)
class VacuumProblem:
    rows: int
    cols: int
    start: Position
    dirty_cells: FrozenSet[Position]

    def __post_init__(self):
        if self.rows <= 0 or self.cols <= 0:
            raise ValueError("Số hàng và số cột phải lớn hơn 0.")
        if not self.is_inside(self.start):
            raise ValueError(f"Vị trí robot {self.start} nằm ngoài ma trận.")
        invalid = [p for p in self.dirty_cells if not self.is_inside(p)]
        if invalid:
            raise ValueError(f"Các ô dirty nằm ngoài ma trận: {invalid}")

    def is_inside(self, position: Position) -> bool:
        x, y = position
        return 1 <= x <= self.cols and 1 <= y <= self.rows

    @property
    def initial_state(self) -> State:
        return self.start, self.dirty_cells


@dataclass
class AStarResult:
    found: bool
    states: List[State]
    actions: List[str]
    step_costs: List[int]
    total_cost: float
    expanded_nodes: int
    generated_nodes: int
    visited_order: List[State]
    message: str


In [ ]:

MOVE_ACTIONS = {
    "MOVE_LEFT":  (-1, 0),
    "MOVE_RIGHT": (1, 0),
    "MOVE_UP":    (0, 1),
    "MOVE_DOWN":  (0, -1),
}


def calculate_step_cost(remaining_dirty_cells: FrozenSet[Position]) -> int:
    """Chi phí hành động = 1 + số ô dirty còn lại sau hành động."""
    return 1 + len(remaining_dirty_cells)


def get_successors(problem: VacuumProblem, state: State):
    robot_position, dirty_cells = state
    successors = []

    # SUCK chỉ hợp lệ khi robot đang đứng trên một ô dirty.
    if robot_position in dirty_cells:
        new_dirty = frozenset(dirty_cells - {robot_position})
        successors.append((
            "SUCK",
            (robot_position, new_dirty),
            calculate_step_cost(new_dirty),
        ))

    # Bốn hành động di chuyển.
    current_x, current_y = robot_position
    for action, (dx, dy) in MOVE_ACTIONS.items():
        new_position = (current_x + dx, current_y + dy)
        if problem.is_inside(new_position):
            successors.append((
                action,
                (new_position, dirty_cells),
                calculate_step_cost(dirty_cells),
            ))

    return successors


def is_goal_state(state: State) -> bool:
    return len(state[1]) == 0



## Hàm heuristic

Với $k$ ô dirty còn lại, robot cần ít nhất $k$ lần `SUCK`. Tổng chi phí hút tối thiểu là:

$$k + (k-1) + \cdots + 1 = \frac{k(k+1)}{2}.$$

Gọi $d_{\min}$ là khoảng cách Manhattan từ robot tới ô dirty gần nhất. Trước lần hút đầu tiên vẫn còn $k$ ô dirty nên mỗi bước di chuyển có chi phí ít nhất $k+1$.

$$h(s)=\frac{k(k+1)}{2}+d_{\min}(k+1).$$

Hàm này không tính phần di chuyển giữa các ô dirty, vì vậy không đánh giá vượt chi phí thực tế.


In [ ]:

def manhattan_distance(a: Position, b: Position) -> int:
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def heuristic(state: State) -> int:
    robot_position, dirty_cells = state
    if not dirty_cells:
        return 0

    k = len(dirty_cells)
    d_min = min(manhattan_distance(robot_position, dirty) for dirty in dirty_cells)
    minimum_suck_cost = k * (k + 1) // 2
    minimum_move_cost = d_min * (k + 1)
    return minimum_suck_cost + minimum_move_cost


In [ ]:

def reconstruct_solution(goal_state: State, parent):
    states = [goal_state]
    actions = []
    step_costs = []
    current = goal_state

    while parent[current] is not None:
        previous, action, step_cost = parent[current]
        states.append(previous)
        actions.append(action)
        step_costs.append(step_cost)
        current = previous

    states.reverse()
    actions.reverse()
    step_costs.reverse()
    return states, actions, step_costs


def a_star_search(problem: VacuumProblem, max_expansions: int = 200_000) -> AStarResult:
    initial_state = problem.initial_state
    tie_breaker = count()
    initial_h = heuristic(initial_state)

    # (f, h, tie_breaker, g, state)
    open_heap = [(initial_h, initial_h, next(tie_breaker), 0, initial_state)]
    g_score = {initial_state: 0}
    parent = {initial_state: None}

    expanded_nodes = 0
    generated_nodes = 0
    visited_order = []

    while open_heap:
        current_f, current_h, _, current_g, current_state = heapq.heappop(open_heap)

        # Bỏ qua bản ghi cũ nếu đã tìm thấy đường tốt hơn.
        if current_g != g_score.get(current_state, math.inf):
            continue

        if is_goal_state(current_state):
            states, actions, step_costs = reconstruct_solution(current_state, parent)
            return AStarResult(
                True, states, actions, step_costs, current_g,
                expanded_nodes, generated_nodes, visited_order,
                "Đã tìm thấy lời giải tối ưu.",
            )

        expanded_nodes += 1
        visited_order.append(current_state)

        if expanded_nodes >= max_expansions:
            return AStarResult(
                False, [], [], [], math.inf,
                expanded_nodes, generated_nodes, visited_order,
                "Đã đạt giới hạn node mở rộng.",
            )

        for action, next_state, step_cost in get_successors(problem, current_state):
            generated_nodes += 1
            tentative_g = current_g + step_cost

            if tentative_g < g_score.get(next_state, math.inf):
                g_score[next_state] = tentative_g
                parent[next_state] = (current_state, action, step_cost)
                next_h = heuristic(next_state)
                heapq.heappush(
                    open_heap,
                    (tentative_g + next_h, next_h, next(tie_breaker), tentative_g, next_state),
                )

    return AStarResult(
        False, [], [], [], math.inf,
        expanded_nodes, generated_nodes, visited_order,
        "Không tìm thấy lời giải.",
    )


In [ ]:

def generate_random_dirty_cells(
    rows: int,
    cols: int,
    number_of_dirty_cells: int,
    start: Position,
    exclude_start: bool = True,
    random_seed: Optional[int] = None,
) -> FrozenSet[Position]:
    positions = [
        (x, y)
        for y in range(1, rows + 1)
        for x in range(1, cols + 1)
        if not (exclude_start and (x, y) == start)
    ]

    if not 0 <= number_of_dirty_cells <= len(positions):
        raise ValueError(f"Số ô dirty phải nằm trong khoảng 0–{len(positions)}.")

    rng = random.Random(random_seed)
    return frozenset(rng.sample(positions, number_of_dirty_cells))


def dataframe_to_dirty_cells(dataframe: pd.DataFrame, rows: int, cols: int) -> FrozenSet[Position]:
    df = dataframe.copy()
    df.columns = [str(column).strip().lower() for column in df.columns]
    if "x" not in df.columns or "y" not in df.columns:
        raise ValueError("File phải có hai cột x và y.")

    coordinates = df[["x", "y"]].dropna()
    coordinates["x"] = pd.to_numeric(coordinates["x"], errors="raise")
    coordinates["y"] = pd.to_numeric(coordinates["y"], errors="raise")

    dirty_cells = set()
    for x, y in coordinates.itertuples(index=False):
        if not float(x).is_integer() or not float(y).is_integer():
            raise ValueError("Tọa độ x, y phải là số nguyên.")
        position = (int(x), int(y))
        if not (1 <= position[0] <= cols and 1 <= position[1] <= rows):
            raise ValueError(f"Tọa độ {position} nằm ngoài ma trận.")
        dirty_cells.add(position)

    return frozenset(dirty_cells)


def load_dirty_cells_from_file(file_path: str, rows: int, cols: int) -> FrozenSet[Position]:
    suffix = Path(file_path).suffix.lower()
    if suffix == ".csv":
        dataframe = pd.read_csv(file_path)
    elif suffix in {".xls", ".xlsx"}:
        dataframe = pd.read_excel(file_path)
    else:
        raise ValueError("Chỉ hỗ trợ file CSV, XLS hoặc XLSX.")
    return dataframe_to_dirty_cells(dataframe, rows, cols)


In [ ]:

def solution_to_dataframe(result: AStarResult) -> pd.DataFrame:
    if not result.found:
        return pd.DataFrame()

    rows = []
    cumulative_cost = 0
    for step, action in enumerate(result.actions, start=1):
        previous_state = result.states[step - 1]
        next_state = result.states[step]
        step_cost = result.step_costs[step - 1]
        cumulative_cost += step_cost
        rows.append({
            "Bước": step,
            "Hành động": action,
            "Vị trí trước": previous_state[0],
            "Vị trí sau": next_state[0],
            "Dirty còn lại": len(next_state[1]),
            "Chi phí bước": step_cost,
            "Chi phí tích lũy": cumulative_cost,
        })
    return pd.DataFrame(rows)


def draw_state(problem: VacuumProblem, state: State, path=None, title=""):
    robot, dirty = state
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.set_xlim(0.5, problem.cols + 0.5)
    ax.set_ylim(0.5, problem.rows + 0.5)
    ax.set_xticks(range(1, problem.cols + 1))
    ax.set_yticks(range(1, problem.rows + 1))
    ax.set_aspect("equal")
    ax.grid(True)

    if path:
        path_x = [position[0] for position in path]
        path_y = [position[1] for position in path]
        ax.plot(path_x, path_y, "-o", linewidth=3, label="Lộ trình")

    if dirty:
        ax.scatter(
            [p[0] for p in dirty], [p[1] for p in dirty],
            marker="s", s=650, label="Dirty (D)", zorder=3,
        )
        for x, y in dirty:
            ax.text(x, y, "D", ha="center", va="center", color="white", fontweight="bold")

    ax.scatter([problem.start[0]], [problem.start[1]], marker="*", s=400, label="Start", zorder=4)
    ax.scatter([robot[0]], [robot[1]], marker="o", s=500, label="Robot", zorder=5)
    ax.text(robot[0], robot[1], "R", ha="center", va="center", color="white", fontweight="bold")

    ax.set_xlabel("Tọa độ x")
    ax.set_ylabel("Tọa độ y")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)

    fig.text(
        0.5,
        0.015,
        f"{PROJECT_TITLE} | Người thực hiện: {STUDENT_NAME}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#4f6475",
    )

    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


## Cấu hình và chạy thử

In [ ]:
print("=" * 72)
print(PROJECT_TITLE.upper())
print(f"Người thực hiện: {STUDENT_NAME}")
print(f"Giảng viên: {LECTURER_NAME}")
print("=" * 72)


# Thay đổi các tham số tại đây.
m = 5                       # Số hàng
n = 7                       # Số cột
robot_start = (1, 1)        # Tọa độ bắt đầu của robot
input_mode = "random"       # "random" hoặc "file"
number_of_dirty_cells = 4
random_seed = 15
file_path = "dirty_cells_sample.csv"  # Dùng khi input_mode = "file"

if input_mode == "random":
    dirty_cells = generate_random_dirty_cells(
        rows=m,
        cols=n,
        number_of_dirty_cells=number_of_dirty_cells,
        start=robot_start,
        exclude_start=True,
        random_seed=random_seed,
    )
else:
    dirty_cells = load_dirty_cells_from_file(file_path, rows=m, cols=n)

problem = VacuumProblem(
    rows=m,
    cols=n,
    start=robot_start,
    dirty_cells=dirty_cells,
)

print("Các ô dirty:", sorted(problem.dirty_cells))
draw_state(problem, problem.initial_state, title=f"Trạng thái ban đầu – {STUDENT_NAME}")


In [ ]:

result = a_star_search(problem, max_expansions=200_000)

print("\nKẾT QUẢ THUẬT TOÁN A*")
print("-" * 72)
print(result.message)
print("Số hành động:", len(result.actions))
print("Tổng chi phí:", result.total_cost)
print("Số node mở rộng:", result.expanded_nodes)
print("Số node đã sinh:", result.generated_nodes)

result_table = solution_to_dataframe(result)
display(result_table)

if result.found:
    path = [state[0] for state in result.states]
    draw_state(
        problem,
        result.states[-1],
        path=path,
        title=f"Lộ trình tối ưu – Tổng chi phí: {result.total_cost}\n{STUDENT_NAME}",
    )


In [ ]:

# Xuất chi tiết lộ trình ra CSV.
if result.found:
    result_table.to_csv("Hoang_Ngoc_Thuy_Thuong_AStar_Result.csv", index=False, encoding="utf-8-sig")
    print("Đã tạo file Hoang_Ngoc_Thuy_Thuong_AStar_Result.csv")


## Kết luận

Chương trình đã đáp ứng đầy đủ các yêu cầu của đề bài:

- Cho phép người dùng chỉ định kích thước ma trận \(A_{m,n}\) và vị trí bắt đầu của robot.
- Các ô dirty có thể được sinh ngẫu nhiên hoặc đọc từ file CSV/Excel.
- Robot có đủ năm hành động: trái, phải, lên, xuống và hút bụi.
- Chi phí mỗi bước được tính đúng theo quy tắc:
  \[
  c = 1 + \left|D_{\text{còn lại sau hành động}}\right|.
  \]
- Thuật toán A* tìm chuỗi hành động làm sạch toàn bộ các ô dirty.
- Chương trình hiển thị lộ trình, chi phí từng bước, chi phí tích lũy và tổng chi phí.
- Gốc tọa độ \((1,1)\) được thể hiện ở góc dưới bên trái.

---

<div style="
    margin-top: 26px;
    padding: 16px 20px;
    border-left: 5px solid #1769aa;
    background: #f2f8fd;
    border-radius: 8px;
">
    <b>Đồ án được thực hiện bởi:</b> Hoàng Ngọc Thủy Thương<br>
    <b>Tên đồ án:</b> Đồ án cuối kỳ – Thuật toán A* cho Robot hút bụi<br>
    <b>Giảng viên hướng dẫn:</b> TS. Nguyễn An Tế
</div>
